<a href="https://colab.research.google.com/github/Saddin13/AulaIA/blob/main/Titanic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [117]:
import pandas as pd

In [118]:
TabelaTreino = pd.read_csv('train.csv')

treino = TabelaTreino.copy()
treino.head(3)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S


In [119]:
TabelaTest = pd.read_csv('test.csv')

test = TabelaTest.copy()
test.head(3)

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q


In [120]:
treino['Ticket_count'] = treino.groupby('Ticket')['PassengerId'].transform('count')
test['Ticket_count']  = test.groupby('Ticket')['PassengerId'].transform('count')

treino = treino.drop(['Name', 'Ticket', 'Cabin'], axis=1)
test  = test.drop(['Name', 'Ticket', 'Cabin'], axis=1)

In [121]:
if 'Sex' in treino.columns:
    treino['isMale'] = treino.Sex.apply(lambda x: 1 if x == 'male' else 0)
    treino = treino.drop('Sex', axis=1)
if 'Sex' in test.columns:
    test['isMale'] = test.Sex.apply(lambda x: 1 if x == 'male' else 0)
    test = test.drop('Sex', axis=1)

In [122]:
treino.head(5)

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare,Embarked,Ticket_count,isMale
0,1,0,3,22.0,1,0,7.2500,S,1,1
1,2,1,1,38.0,1,0,71.2833,C,1,0
2,3,1,3,26.0,0,0,7.9250,S,1,0
3,4,1,1,35.0,1,0,53.1000,S,2,0
4,5,0,3,35.0,0,0,8.0500,S,1,1


In [123]:
if 'Embarked' in treino.columns:
    treino['Embarked'] = treino.Embarked.apply (lambda x: 0 if x == 'S' else (1 if x == 'C' else 2))
if 'Embarked' in test.columns:
    test['Embarked'] = test.Embarked.apply (lambda x: 0 if x == 'S' else (1 if x == 'C' else 2))

In [124]:
treino = pd.get_dummies(treino, columns=['Embarked'], prefix='Emb')
test   = pd.get_dummies(test,   columns=['Embarked'], prefix='Emb')

In [125]:
treino['Fare_per_person'] = treino['Fare'] / treino['Ticket_count']
treino = treino.drop(['Fare'], axis=1)
test['Fare_per_person']   = test['Fare']   / test['Ticket_count']
test = test.drop(['Fare'], axis=1)

In [126]:
treino['Age'] = treino['Age'].fillna(treino['Age'].median())
test['Age']   = test['Age'].fillna(treino['Age'].median())

In [127]:
treino.head(3)

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Ticket_count,isMale,Emb_0,Emb_1,Emb_2,Fare_per_person
0,1,0,3,22.0,1,0,1,1,True,False,False,7.2500
1,2,1,1,38.0,1,0,1,0,False,True,False,71.2833
2,3,1,3,26.0,0,0,1,0,True,False,False,7.9250


In [132]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, confusion_matrix

# Separando X e y
X = treino.drop(['PassengerId', 'Survived'], axis=1)
y = treino.Survived

# Split treino/validação
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.33, random_state=42)

# Parâmetros a testar
params = {
    'n_estimators':      [100, 200, 300],
    'max_depth':         [4, 5, 6, 7],
    'min_samples_split': [2, 4, 6],
    'min_samples_leaf':  [1, 2, 3]
}

# Grid Search
grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    params,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid.fit(X_train, y_train)

# Resultados
print('Melhores parâmetros:', grid.best_params_)
print('Melhor acurácia (cv):', f'{grid.best_score_:.2%}')

# Avaliando na validação
clf_rf    = grid.best_estimator_
y_pred_rf = clf_rf.predict(X_val)
print('Acurácia na validação:', f'{accuracy_score(y_val, y_pred_rf):.2%}')
print('Matriz de confusão:\n', confusion_matrix(y_val, y_pred_rf))

# Previsão no test
X_teste = test.drop('PassengerId', axis=1)
y_pred  = clf_rf.predict(X_teste)

# Submissão
base_envio = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Survived':    y_pred
})

base_envio.to_csv('resultados_rf_tuned.csv', index=False)
print(base_envio.head())

KeyboardInterrupt: 